# Assignment 10
Q. Develop a Machine Translation system to translate public information content between English and any Indian language.

Step 1: Install Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')
!pip install transformers datasets sentencepiece sacrebleu evaluate -q

Step 2: Import Libraries

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset
import numpy as np
import evaluate

Step 3: Load Dataset (English → Hindi)

In [ ]:
from huggingface_hub import login
login() # This will prompt you to enter your Hugging Face token
dataset = load_dataset("cfilt/iitb-english-hindi")

# Reduce size for faster training (important for Colab)
dataset["train"] = dataset["train"].select(range(10000))
dataset["validation"] = dataset["validation"].select(range(500))

Step 4: Load Pretrained Model

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Step 5: Preprocess Data

In [ ]:
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["hi"] for ex in examples["translation"]]

    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    labels = tokenizer(targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

Step 6: Evaluation Metric (BLEU Score)

In [ ]:
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    preds = np.argmax(preds, axis=-1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])

    return {"bleu": result["score"]}

Step 7: Training Setup

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt-en-hi",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,  # increase to 3–5 for better results
    predict_with_generate=True,
    logging_dir="./logs",
)

Step 8: Trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

In [ ]:
def translate(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example
print(translate("Government services are available offline"))